In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print("GPU available:", tf.config.list_physical_devices('GPU'))

2026-06-08 05:08:11.147293: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780895291.356952      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780895291.416836      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780895291.902568      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780895291.902619      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780895291.902625      57 computation_placer.cc:177] computation placer alr

GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [2]:
import os

base_dir = '/kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/real_vs_fake/real-vs-fake'
for split in ['train', 'valid','test']:
    for label in ['real', 'fake']:
        path = os.path.join(base_dir, split, label)
        count = len(os.listdir(path))
        print(f"{split}/{label} : {count} anhh")

train/real : 50000 anhh
train/fake : 50000 anhh
valid/real : 10000 anhh
valid/fake : 10000 anhh
test/real : 10000 anhh
test/fake : 10000 anhh


In [3]:
IMG_SIZE = 224
BATCH_SIZE = 64

# Augmentation cho train 
train_datagen = ImageDataGenerator(
    rescale=1./255, # normalize pixel ve [0,1]
    rotation_range = 30, #xoay toi da 30 do
    width_shift_range=0.2,#dich ngang toi da 20
    height_shift_range=0.2,# dich doc toi da 20
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip = True,
    brightness_range=[0.8,1.2],
    fill_mode='nearest' #fill pixel thieu bang nearest neighbor
)

#valid/test chi normalize, khong augment
val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    os.path.join(base_dir, 'train'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size = BATCH_SIZE,
    class_mode = 'binary' #real 1, fake =0
)

val_gen = val_datagen.flow_from_directory(
    os.path.join(base_dir, 'valid'),
    target_size = (IMG_SIZE, IMG_SIZE),
    batch_size = BATCH_SIZE,
    class_mode = 'binary'
)

test_gen = val_datagen.flow_from_directory(
    os.path.join(base_dir, 'test'),
    target_size = (IMG_SIZE, IMG_SIZE),
    batch_size = BATCH_SIZE,
    class_mode = 'binary',
    shuffle=False
)

print("Train:", train_gen.samples,"anh")
print("Valid:", val_gen.samples,"anh")
print("Class indices:", train_gen.class_indices)

Found 100000 images belonging to 2 classes.
Found 20000 images belonging to 2 classes.
Found 20000 images belonging to 2 classes.
Train: 100000 anh
Valid: 20000 anh
Class indices: {'fake': 0, 'real': 1}


In [4]:
def build_cnn_block(x, filters, dropout_rate=0.25):
    """
    1 conv block = Conv2D → BatchNorm → ReLU → MaxPool → Dropout
    filters: số filter tăng dần 32→64→128→256
    """
    x = layers.Conv2D(
        filters,
        kernel_size=(3, 3),
        padding='same',
        kernel_initializer='he_normal',   # paper dùng He normal init
        kernel_regularizer=tf.keras.regularizers.l2(1e-4)  # L2 reg
    )(x)
    x = layers.BatchNormalization()(x)    # ổn định training
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)  # giảm spatial dim
    x = layers.Dropout(dropout_rate)(x)
    return x

# Build full CNN backbone
inputs = layers.Input(shape=(224, 224, 3))

x = build_cnn_block(inputs, filters=32)   # 224→112
x = build_cnn_block(x, filters=64)        # 112→56
x = build_cnn_block(x, filters=128)       # 56→28
x = build_cnn_block(x, filters=256)       # 28→14

print("Shape sau 4 conv blocks:", x.shape)
# Kết quả mong đợi: (None, 14, 14, 256)

I0000 00:00:1780895445.146211      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1780895445.152358      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Shape sau 4 conv blocks: (None, 14, 14, 256)


In [5]:
def build_mhsa_block(x):
    """
    Sau conv blocks: Reshape → Dropout → MHSA → Residual → LayerNorm
    Đây là phần paper gọi là 'sequential integration'
    """
    # Bước 1: Reshape từ (14, 14, 256) → (196, 256)
    # MHSA cần input dạng sequence, không phải spatial grid
    seq_len = x.shape[1] * x.shape[2]   # 14×14 = 196
    x_reshaped = layers.Reshape((seq_len, 256))(x)  # (None, 196, 256)

    # Bước 2: Dropout trước attention (paper dùng 0.3)
    x_drop = layers.Dropout(0.3)(x_reshaped)

    # Bước 3: Multi-Head Self-Attention
    # num_heads=4, key_dim=64 — đúng theo paper Table 2
    attn_output = layers.MultiHeadAttention(
        num_heads=4,
        key_dim=64,
        dropout=0.2          # dropout bên trong attention (paper: 0.2)
    )(x_drop, x_drop)        # self-attention: query=key=value=x_drop

    # Bước 4: Residual connection — cộng input gốc vào output attention
    # Giúp giữ lại thông tin low-level từ CNN, tránh mất mát
    x_residual = layers.Add()([x_reshaped, attn_output])

    # Bước 5: Layer Normalization
    x_norm = layers.LayerNormalization()(x_residual)

    return x_norm

# Gắn MHSA vào sau CNN backbone
x = build_mhsa_block(x)

# Global Average Pooling: (196, 256) → (256,)
# Gộp toàn bộ sequence thành 1 vector đặc trưng
x = layers.GlobalAveragePooling1D()(x)

# Dense layers — phân loại cuối
x = layers.Dense(128)(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)

x = layers.Dense(64)(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)

# Output layer: sigmoid vì binary classification
outputs = layers.Dense(1, activation='sigmoid')(x)

# Tạo model
model = Model(inputs=inputs, outputs=outputs)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 224, 224,  │        896 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 224, 224,  │        128 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 224, 224,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 112, 112,  │          0 │ activation[0][0]  │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 112, 112,  │          0 │ max_pooling2d[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 112, 112,  │     18,496 │ dropout[0][0]     │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 112, 112,  │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 112, 112,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 56, 56,    │          0 │ activation_1[0][… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 56, 56,    │          0 │ max_pooling2d_1[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 56, 56,    │     73,856 │ dropout_1[0][0]   │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 56, 56,    │        512 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 56, 56,    │          0 │ batch_normalizat… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 28, 28,    │          0 │ activation_2[0][… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 28, 28,    │          0 │ max_pooling2d_2[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 28, 28,    │    295,168 │ dropout_2[0][0] 

 Total params: 696,001 (2.66 MB)

 Trainable params: 694,657 (2.65 MB)

 Non-trainable params: 1,344 (5.25 KB)

In [6]:
import tensorflow.keras.backend as K

def focal_loss(alpha=0.25, gamma=2.0):
    """
    Focal Loss — paper dùng alpha=0.25, gamma=2.0
    
    Tại sao không dùng Binary Cross-Entropy thông thường?
    BCE phạt mọi sample như nhau. Focal Loss phạt nặng hơn
    những sample model đang predict sai với confidence cao.
    gamma=2 nghĩa là sample dễ (confidence cao, đúng) 
    sẽ bị giảm loss đi rất nhiều → model tập trung vào hard samples.
    """
    def loss_fn(y_true, y_pred):
        y_pred = K.clip(y_pred, 1e-7, 1 - 1e-7)  # tránh log(0)
        
        # Binary cross entropy cơ bản
        bce = -y_true * K.log(y_pred) - (1 - y_true) * K.log(1 - y_pred)
        
        # Tính p_t: xác suất predict đúng class
        p_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        
        # Focal weight: (1 - p_t)^gamma
        # p_t cao (dễ) → weight gần 0 → loss nhỏ
        # p_t thấp (khó) → weight gần 1 → loss giữ nguyên
        focal_weight = K.pow(1 - p_t, gamma)
        
        # Alpha cân bằng 2 class
        alpha_t = y_true * alpha + (1 - y_true) * (1 - alpha)
        
        return K.mean(alpha_t * focal_weight * bce)
    
    return loss_fn

# Compile model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=focal_loss(alpha=0.25, gamma=2.0),
    metrics=['accuracy',
             tf.keras.metrics.AUC(name='auc'),
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

print("Model compiled OK")

Model compiled OK


In [7]:
CHECKPOINT_PATH = '/kaggle/working/best_model.keras'
BACKUP_PATH = '/kaggle/working/backup_epoch.keras'

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_auc', patience=7,
        restore_best_weights=True,
        mode='max', verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_auc', factor=0.5,
        patience=3, min_lr=1e-6,
        mode='max', verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        CHECKPOINT_PATH, monitor='val_auc',
        save_best_only=True, mode='max', verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        BACKUP_PATH, save_best_only=False, verbose=0
    )
]

history = model.fit(
    train_gen,
    epochs=10,          # chỉ train đến epoch 10
    initial_epoch=0,
    validation_data=val_gen,
    callbacks=callbacks,
    verbose=1
)

print(f"\nBest val_auc: {max(history.history['val_auc']):.4f}")
print(f"Best val_accuracy: {max(history.history['val_accuracy']):.4f}")

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10


I0000 00:00:1780745535.458158     150 service.cc:152] XLA service 0x7b0d5820bba0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1780745535.458207     150 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1780745535.458230     150 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1780745536.849646     150 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1780745554.862332     150 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


KeyboardInterrupt: 

In [ ]:
# ============================================
START_EPOCH = 50   # thay số này mỗi lần
END_EPOCH = 53     # thay số này mỗi lần
# ============================================

CHECKPOINT_PATH = '/kaggle/working/best_model.keras'
BACKUP_PATH = '/kaggle/working/backup_epoch.keras'

# Load best model
model = tf.keras.models.load_model(
    '/kaggle/input/models/manht05/last/keras/default/13/best.keras',
    custom_objects={'loss_fn': focal_loss()}
)

# Tính lr giảm dần theo số lần chạy
lr = 1e-4 * (0.8 ** (START_EPOCH // 10))
print(f"Resume epoch {START_EPOCH}→{END_EPOCH}, lr={lr:.6f}")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
    loss=focal_loss(alpha=0.25, gamma=2.0),
    metrics=['accuracy',
             tf.keras.metrics.AUC(name='auc'),
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_auc', patience=7,
        restore_best_weights=True,
        mode='max', verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_auc', factor=0.5,
        patience=3, min_lr=1e-6,
        mode='max', verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        CHECKPOINT_PATH, monitor='val_auc',
        save_best_only=True, mode='max', verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        BACKUP_PATH, save_best_only=False, verbose=0
    )
]

history = model.fit(
    train_gen,
    epochs=END_EPOCH,
    initial_epoch=START_EPOCH,
    validation_data=val_gen,
    callbacks=callbacks,
    verbose=1
)

print(f"\n--- Epoch {START_EPOCH}→{END_EPOCH} xong ---")
print(f"Best val_auc:      {max(history.history['val_auc']):.4f}")
print(f"Best val_accuracy: {max(history.history['val_accuracy']):.4f}")
print(f"Nhớ Save Version trước khi tắt!")

Resume epoch 50→53, lr=0.000033
Epoch 51/53
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 0s 999ms/step - accuracy: 0.9217 - auc: 0.9835 - loss: 0.0268 - precision: 0.9646 - recall: 0.8757
Epoch 51: val_auc improved from -inf to 0.96458, saving model to /kaggle/working/best_model.keras
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 1657s 1s/step - accuracy: 0.9217 - auc: 0.9835 - loss: 0.0268 - precision: 0.9646 - recall: 0.8757 - val_accuracy: 0.8778 - val_auc: 0.9646 - val_loss: 0.0893 - val_precision: 0.8201 - val_recall: 0.9681 - learning_rate: 3.2768e-05
Epoch 52/53
 763/1563 ━━━━━━━━━━━━━━━━━━━━ 13:05 982ms/step - accuracy: 0.9254 - auc: 0.9844 - loss: 0.0264 - precision: 0.9673 - recall: 0.8809

In [12]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_curve, auc)
import seaborn as sns

# Load best model
model = tf.keras.models.load_model(
    '/kaggle/input/models/manht05/last/keras/default/13/best.keras',
    custom_objects={'loss_fn': focal_loss()}
)

# Predict trên test set
print("Đang predict...")
y_pred_prob = model.predict(test_gen, verbose=1)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()
y_true = test_gen.classes

# --- Thử các threshold khác nhau ---
print("\n--- Threshold Analysis ---")
for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:
    y_pred_t = (y_pred_prob > threshold).astype(int).flatten()
    from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
    acc = accuracy_score(y_true, y_pred_t)
    prec = precision_score(y_true, y_pred_t)
    rec = recall_score(y_true, y_pred_t)
    f1 = f1_score(y_true, y_pred_t)
    print(f"threshold={threshold:.1f} | acc={acc:.3f} | prec={prec:.3f} | rec={rec:.3f} | f1={f1:.3f}")

Đang predict...
313/313 ━━━━━━━━━━━━━━━━━━━━ 65s 202ms/step

--- Threshold Analysis ---
threshold=0.3 | acc=0.781 | prec=0.698 | rec=0.993 | f1=0.820
threshold=0.4 | acc=0.822 | prec=0.743 | rec=0.985 | f1=0.847
threshold=0.5 | acc=0.857 | prec=0.790 | rec=0.974 | f1=0.872
threshold=0.6 | acc=0.883 | prec=0.833 | rec=0.957 | f1=0.891
threshold=0.7 | acc=0.901 | prec=0.879 | rec=0.929 | f1=0.903


In [ ]:
from sklearn.metrics import confusion_matrix, roc_curve, auc, classification_report
import seaborn as sns

THRESHOLD = 0.7
y_pred_final = (y_pred_prob > THRESHOLD).astype(int).flatten()

# --- Confusion Matrix ---
cm = confusion_matrix(y_true, y_pred_final)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Fake', 'Real'],
            yticklabels=['Fake', 'Real'])
plt.title('Confusion Matrix (threshold=0.7)')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix.png', dpi=150)
plt.show()

# --- ROC Curve ---
fpr, tpr, _ = roc_curve(y_true, y_pred_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', lw=2,
         label=f'ROC Curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Deepfake Detection')
plt.legend()
plt.tight_layout()
plt.savefig('/kaggle/working/roc_curve.png', dpi=150)
plt.show()

# --- Classification Report ---
print("\n--- Classification Report (threshold=0.7) ---")
print(classification_report(y_true, y_pred_final,
                            target_names=['Fake', 'Real']))